# 12_longitudinal_tracking

Apply the counterfactual at the 2019 baseline and track the cohort forward to
2024. Two complementary views:

1. **Observed tracking.** Among the 2019 HTN-free cohort, split by whether BMI
   was actually reduced (>=1 kg/m2) over 2019->2020, then follow cumulative
   hypertension incidence through 2021-2024. Crude trajectories diverge (the
   reducers do *worse*), but this is selection: reducers start older and
   heavier. After adjusting for age and baseline BMI, the odds ratio sits at
   ~1.0 at every horizon (2-5 years) - the null persists long-term, directly
   testing the earlier "prediction window too short" hypothesis.

2. **Model simulation.** Apply a BMI reduction to the 2019 features and read off
   the model's predicted risk. The model *expects* a ~15% relative risk drop
   for a 3 kg/m2 reduction - but the observed long-term data show no such
   effect. This gap between what the counterfactual model promises and what the
   longitudinal data deliver is the central message of the study.

In [1]:
# 12_longitudinal_tracking.ipynb
# Observed long-term tracking of the 2019 baseline cohort + model simulation.

import os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

ROOT = os.path.abspath("..")
DATA = os.path.join(ROOT, "data")
FIG  = os.path.join(ROOT, "results", "figures")
TAB  = os.path.join(ROOT, "results", "tables")

panel = pd.read_parquet(os.path.join(DATA, "khp_panel_long.parquet"))
adult = panel[panel.age >= 19].copy()

# 2019 baseline cohort: HTN-free, complete targets
base = adult[adult.year == 2019][["PIDWON","BMI","HTN","age","SEX",
                                  "smoke_cur","exer_reg","walk_days"]]
base = base[base["HTN"] == 0].dropna(subset=["BMI","age","smoke_cur","exer_reg"])

# Treatment defined over 2019->2020; require HTN-free at 2020 too
d20 = adult[adult.year==2020][["PIDWON","BMI","HTN"]].rename(columns={"BMI":"BMI20","HTN":"HTN20"})
b2 = base.merge(d20, on="PIDWON")
b2 = b2[b2["HTN20"] == 0].copy()
b2["achieved"] = ((b2["BMI20"] - b2["BMI"]) <= -1.0).astype(int)
b2.to_parquet(os.path.join(DATA, "cohort2019.parquet"))
print(f"2019 cohort with treatment defined: {len(b2)} "
      f"(achieved {b2['achieved'].sum()})")

2019 cohort with treatment defined: 6713 (achieved 989)


In [2]:
# (1) Observed tracking: cumulative HTN incidence by year, crude + adjusted.
htn_by_year = {yr: adult[adult.year==yr].set_index("PIDWON")["HTN"]
               for yr in [2020,2021,2022,2023,2024]}
b2["female"] = (b2["SEX"] == 2).astype(int)

rows, adj = [], []
for yr in [2021, 2022, 2023, 2024]:
    sub = b2.copy()
    hit = np.zeros(len(sub), dtype=bool)
    for y2 in range(2021, yr+1):
        hit = hit | (sub["PIDWON"].map(htn_by_year[y2]) == 1).values
    sub["cum_htn"] = hit.astype(int)
    obs = sub["PIDWON"].map(htn_by_year[yr]).notna().values
    sub = sub[obs]

    g = sub.groupby("achieved")["cum_htn"].mean() * 100
    rows.append({"year": yr, "ach": g.get(1, np.nan), "noach": g.get(0, np.nan),
                 "n_ach": int((sub.achieved==1).sum())})

    m = smf.logit("cum_htn ~ achieved + BMI + age + female", data=sub).fit(disp=0)
    OR = np.exp(m.params["achieved"]); ci = np.exp(m.conf_int().loc["achieved"])
    adj.append({"year": yr, "OR": OR, "lo": ci[0], "hi": ci[1],
                "p": m.pvalues["achieved"]})

R = pd.DataFrame(rows); A = pd.DataFrame(adj)
R.to_parquet(os.path.join(DATA, "longterm_observed.parquet"))
A.round(3).to_csv(os.path.join(TAB, "table15_longterm_observed.csv"), index=False)

for _, r in A.iterrows():
    print(f"{int(r['year'])} ({int(r['year'])-2019}yr): "
          f"adjusted OR={r['OR']:.3f} (95% CI {r['lo']:.3f}-{r['hi']:.3f}), p={r['p']:.3f}")

2021 (2yr): adjusted OR=1.236 (95% CI 0.811-1.886), p=0.324
2022 (3yr): adjusted OR=0.921 (95% CI 0.677-1.255), p=0.604
2023 (4yr): adjusted OR=0.974 (95% CI 0.743-1.275), p=0.845
2024 (5yr): adjusted OR=0.982 (95% CI 0.772-1.248), p=0.879


In [3]:
# (2) Model simulation: predicted risk under counterfactual BMI reductions.
htn = pd.read_parquet(os.path.join(DATA, "htn_analysis.parquet"))
htn["female"] = (htn["SEX"] == 2).astype(float)
FEATS = ["BMI","age","female","smoke_cur","exer_reg","walk_days"]
Xtr, Xte, ytr, yte = train_test_split(htn[FEATS].astype(float), htn["incident"].astype(int),
                                      test_size=0.25, random_state=42, stratify=htn["incident"])
clf = RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=30,
        class_weight="balanced", random_state=42).fit(Xtr, ytr)

b2["walk_days"] = b2["walk_days"].fillna(b2["walk_days"].median())
Xreal = b2[FEATS].astype(float)
reds = np.arange(0, 8.1, 0.5)
sim = pd.DataFrame({"reduction": reds,
    "pred_risk": [clf.predict_proba(
        Xreal.assign(BMI=np.maximum(Xreal["BMI"]-r, 18.5)))[:,1].mean()*100
        for r in reds]})
sim.to_parquet(os.path.join(DATA, "simulation_curve.parquet"))

base_risk = sim.loc[sim.reduction==0, "pred_risk"].values[0]
r3 = sim.loc[sim.reduction==3, "pred_risk"].values[0]
print(f"Model-expected risk: {base_risk:.2f}% -> {r3:.2f}% at BMI -3 "
      f"({100*(base_risk-r3)/base_risk:.0f}% relative drop)")
print("Observed long-term adjusted OR ~1.0 -> the expected drop does not materialize.")

Model-expected risk: 37.71% -> 32.09% at BMI -3 (15% relative drop)
Observed long-term adjusted OR ~1.0 -> the expected drop does not materialize.


In [4]:
# Figures 8-10.
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"font.size": 11, "axes.edgecolor": "0.3",
                     "grid.color": "0.85", "savefig.dpi": 600})
yrs = [2, 3, 4, 5]

# Fig 8: observed cumulative-incidence trajectories (crude).
fig, ax = plt.subplots(figsize=(5.5, 3.8))
ax.plot(yrs, R["ach"],   marker="o", color="0.15", lw=1.6, label="BMI reduction achieved")
ax.plot(yrs, R["noach"], marker="s", color="0.55", lw=1.6, ls="--", label="Not achieved")
ax.set_xlabel("Years since 2019 baseline"); ax.set_ylabel("Cumulative HTN incidence (%)")
ax.set_xticks(yrs); ax.legend(frameon=False)
fig.savefig(os.path.join(FIG, "fig8_longterm_trajectory.png"), dpi=600, bbox_inches="tight")
fig.savefig(os.path.join(FIG, "fig8_longterm_trajectory.pdf"), bbox_inches="tight")
plt.close(fig)

# Fig 9: adjusted OR over follow-up horizon.
fig, ax = plt.subplots(figsize=(5.5, 3.8))
ax.errorbar(A["year"]-2019, A["OR"], yerr=[A["OR"]-A["lo"], A["hi"]-A["OR"]],
            fmt="o", color="0.15", ecolor="0.5", capsize=3, ms=6, lw=1.4)
ax.axhline(1.0, color="0.4", ls=":", lw=1)
ax.set_xlabel("Years since 2019 baseline"); ax.set_ylabel("Adjusted OR (95% CI)")
ax.set_xticks(yrs)
fig.savefig(os.path.join(FIG, "fig9_longterm_adjusted_or.png"), dpi=600, bbox_inches="tight")
fig.savefig(os.path.join(FIG, "fig9_longterm_adjusted_or.pdf"), bbox_inches="tight")
plt.close(fig)

# Fig 10: model simulation curve.
fig, ax = plt.subplots(figsize=(5.5, 3.8))
ax.plot(sim["reduction"], sim["pred_risk"], marker="o", color="0.2", lw=1.6, ms=4)
ax.set_xlabel("Counterfactual BMI reduction (kg/m2)")
ax.set_ylabel("Model-predicted risk (%)")
fig.savefig(os.path.join(FIG, "fig10_simulation_curve.png"), dpi=600, bbox_inches="tight")
fig.savefig(os.path.join(FIG, "fig10_simulation_curve.pdf"), bbox_inches="tight")
plt.close(fig)
print("Figures 8, 9, 10 saved (png + pdf).")

Figures 8, 9, 10 saved (png + pdf).
